# Power-Law vs. Pareto Tail Shape in Dependency Distance

## Demo Notebook

This notebook implements a **peaks-over-threshold extreme-value analysis** of sentence-normalized dependency distances across UD treebanks, comparing:
- **GPD shape parameter (xi)** — our method (Generalized Pareto Distribution via MLE)
- **Power-law exponent (alpha)** — baseline (Clauset et al. 2009)

### What this artifact does
- Loads dependency distance data from UD treebanks
- Fits extreme-value models (GPD and power-law) above user-specified thresholds
- Computes bootstrap confidence intervals for both tail indices
- Analyzes matched spoken/written language pairs
- Runs mixed-effects models predicting register from tail shape
- Tests whether spoken language minimizes dependency distances more than written language

**This demo uses a small curated dataset to run in under 10 minutes. Scale up the parameters in the Config cell to reproduce the full analysis.**

In [ ]:
import subprocess, sys
def _pip(*a): subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *a])

# Non-Colab packages: always install
_pip('loguru==0.7.3')

# Pre-installed on Colab (install locally to match Colab env)
if 'google.colab' not in sys.modules:
    _pip('numpy==2.0.2', 'pandas==2.2.2', 'scipy==1.16.3', 'statsmodels==0.14.6', 'matplotlib==3.10.0')

In [ ]:
from __future__ import annotations
import gc
import json
import numpy as np
import pandas as pd
from scipy import stats
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt
from loguru import logger
import sys

logger.remove()
logger.add(sys.stdout, level="INFO", format="{time:HH:mm:ss}|{level:<7}|{message}")

In [ ]:
GITHUB_DATA_URL = "https://raw.githubusercontent.com/ai-inventor-papers/ai-invention-544c17-language-minimizes-dependency-distance/main/round-2/experiment-1/demo/mini_demo_data.json"
import urllib.request
import os

def load_data():
    """Load demo data from GitHub (with local fallback for development)."""
    try:
        with urllib.request.urlopen(GITHUB_DATA_URL, timeout=10) as response:
            return json.loads(response.read().decode())
    except Exception as e:
        logger.debug(f"GitHub fetch failed ({e}); trying local fallback")
    
    if os.path.exists("mini_demo_data.json"):
        with open("mini_demo_data.json") as f:
            return json.load(f)
    
    raise FileNotFoundError("Could not load mini_demo_data.json from GitHub or local path")

In [ ]:
data = load_data()
logger.info(f"Loaded {len(data['datasets'][0]['examples'])} examples from {len(set(e.get('metadata_treebank_id') for e in data['datasets'][0]['examples']))} treebanks")

## Configuration

Set analysis parameters here. **Demo defaults are minimal for speed; scale up to reproduce the full analysis.**

In [ ]:
# ===== DEMO CONFIGURATION (minimal scale for quick run) =====
SEED = 20260907
B_BOOTSTRAP = 50  # demo: reduced from 300 (production: 1000)
PRIMARY_THRESHOLD = 0.75
THRESHOLDS = [0.75, 0.80]  # demo: 2 thresholds (production: 3)
MIN_ARCS_PRIMARY = 10  # demo: minimal (production: 1000)

# ===== PRODUCTION DEFAULTS (uncomment to run full analysis) =====
# B_BOOTSTRAP = 300  # Full bootstrap resamples
# THRESHOLDS = [0.75, 0.80, 0.90]  # All three thresholds
# MIN_ARCS_PRIMARY = 1000  # Production minimum

logger.info(f"Config: B_BOOTSTRAP={B_BOOTSTRAP}, THRESHOLDS={THRESHOLDS}, PRIMARY_THRESHOLD={PRIMARY_THRESHOLD}")

## Phase 1: Load and Parse Data

Convert raw examples into a structured sentence-level DataFrame with parsed outputs.

In [ ]:
def examples_to_frame(examples):
    rows = []
    for ex in examples:
        try:
            out = json.loads(ex["output"])
        except (json.JSONDecodeError, KeyError):
            continue
        norm = out.get("normalized_distances", [])
        gb_raw = ex.get("metadata_grambank_features")
        gb_composite = np.nan
        if gb_raw:
            try:
                gb = json.loads(gb_raw)
                vals = [float(v) for v in gb.values() if v not in (None, "?", "")]
                if vals:
                    gb_composite = float(np.mean(vals))
            except (json.JSONDecodeError, ValueError, TypeError):
                pass
        rows.append({
            "treebank_id": ex.get("metadata_treebank_id"),
            "language": ex.get("metadata_language"),
            "family": ex.get("metadata_language_family"),
            "register": ex.get("metadata_register"),
            "sentence_length": ex.get("metadata_sentence_length"),
            "head_finality_ratio": ex.get("metadata_head_finality_ratio"),
            "grambank_composite": gb_composite,
            "normalized_distances": norm,
            "n_arcs": len(norm),
        })
    df = pd.DataFrame(rows)
    logger.info(f"Built frame: {len(df)} sentences, {df['treebank_id'].nunique()} treebanks")
    return df

sent_df = examples_to_frame(data["datasets"][0]["examples"])
print(sent_df.head())

## Phase 2: Aggregate to Treebank Level

Pool dependency distances across all sentences in each treebank.

In [ ]:
def build_treebank_frame(sent_df):
    rows = []
    for tb_id, grp in sent_df.groupby("treebank_id"):
        distances = np.concatenate([np.asarray(d, dtype=float) for d in grp["normalized_distances"]])
        rows.append({
            "treebank_id": tb_id,
            "language": grp["language"].iloc[0],
            "family": grp["family"].iloc[0],
            "register": grp["register"].iloc[0],
            "n_sentences": len(grp),
            "n_arcs": int(distances.size),
            "head_finality_ratio": float(grp["head_finality_ratio"].mean()),
            "grambank_composite": float(grp["grambank_composite"].mean()) if grp["grambank_composite"].notna().any() else np.nan,
            "_distances": distances,
        })
    return pd.DataFrame(rows)

tb_df = build_treebank_frame(sent_df)
del sent_df
gc.collect()
print(tb_df[["treebank_id", "language", "register", "n_arcs"]])

## Phase 3: Extreme Value Theory Fitting Functions

Fit Generalized Pareto Distribution (xi) and power-law exponent (alpha) to tail exceedances.

In [ ]:
def fit_gpd_point(exceedances):
    """Fit Generalized Pareto (loc=0) via MLE, return (xi, sigma, loglik)."""
    mean_ex = exceedances.mean()
    var_ex = exceedances.var()
    xi0 = 0.5 * (1 - mean_ex**2 / var_ex) if var_ex > 0 else 0.1
    xi0 = float(np.clip(xi0, -0.4, 0.9))
    sigma0 = max(mean_ex * (1 - xi0), 1e-6)
    try:
        c, loc, scale = stats.genpareto.fit(exceedances, xi0, floc=0)
        ll = float(np.sum(stats.genpareto.logpdf(exceedances, c, loc=0, scale=scale)))
        return float(c), float(scale), ll
    except Exception as e:
        logger.warning(f"GPD MLE failed ({e}); falling back to method-of-moments")
        return xi0, sigma0, float("nan")

def fit_powerlaw_alpha(exceedances_from_xmin, xmin):
    """Clauset et al. (2009) discrete-continuous MLE for power law."""
    x = exceedances_from_xmin[exceedances_from_xmin > 0]
    n = x.size
    if n < 2 or xmin <= 0:
        return float("nan"), float("nan")
    s = np.sum(np.log(x / xmin))
    if s <= 0:
        return float("nan"), float("nan")
    alpha = 1.0 + n / s
    se = (alpha - 1.0) / np.sqrt(n)
    return float(alpha), float(se)

def bootstrap_ci(values, xmin, rng, b):
    """Bootstrap confidence intervals for xi and alpha."""
    n = values.size
    xi_boot = np.empty(b)
    alpha_boot = np.empty(b)
    for i in range(b):
        idx = rng.integers(0, n, size=n)
        resample = values[idx]
        exc = resample - xmin
        xi_b, _, _ = fit_gpd_point(exc)
        xi_boot[i] = xi_b
        a_b, _ = fit_powerlaw_alpha(resample, xmin)
        alpha_boot[i] = a_b
    return xi_boot, alpha_boot

logger.info("EVT fitting functions defined")

## Phase 4: Fit EVT Models Across All Treebanks and Thresholds

For each treebank and threshold combination, fit GPD and power-law models with bootstrap CIs.

In [ ]:
def analyze_treebank_threshold(treebank_id, distances, quantile, seed):
    """Fit EVT models for one (treebank, threshold) pair."""
    distances = np.asarray(distances, dtype=float)
    distances = distances[distances > 0]
    t = float(np.quantile(distances, quantile))
    above = distances[distances > t]
    exceed = above - t
    n_exc = exceed.size
    if n_exc < 3:  # demo: minimal requirement
        return {
            "treebank_id": treebank_id, "quantile": quantile, "threshold": t,
            "n_exceedances": n_exc, "insufficient_exceedances": True,
        }
    xi, sigma, ll = fit_gpd_point(exceed)
    alpha, alpha_se = fit_powerlaw_alpha(above, t)
    result = {
        "treebank_id": treebank_id, "quantile": quantile, "threshold": t,
        "n_exceedances": n_exc, "insufficient_exceedances": False,
        "xi_point": xi, "sigma_point": sigma, "gpd_loglik": ll,
        "alpha_point": alpha, "alpha_se": alpha_se,
    }
    # Bootstrap only for demo if we have enough exceedances
    if n_exc >= 5:
        rng = np.random.default_rng(seed)
        xi_boot, alpha_boot = bootstrap_ci(above, t, rng, B_BOOTSTRAP)
        xi_boot_clean = xi_boot[np.isfinite(xi_boot)]
        alpha_boot_clean = alpha_boot[np.isfinite(alpha_boot)]
        if xi_boot_clean.size > 2:
            result["xi_ci_lower"] = float(np.percentile(xi_boot_clean, 2.5))
            result["xi_ci_upper"] = float(np.percentile(xi_boot_clean, 97.5))
        if alpha_boot_clean.size > 2:
            result["alpha_ci_lower"] = float(np.percentile(alpha_boot_clean, 2.5))
            result["alpha_ci_upper"] = float(np.percentile(alpha_boot_clean, 97.5))
    return result

# Run analysis for all treebanks and thresholds
fit_results = {}
for _, row in tb_df.iterrows():
    for q in THRESHOLDS:
        seed_val = SEED + hash(row["treebank_id"]) % 10_000 + int(q * 100)
        res = analyze_treebank_threshold(row["treebank_id"], row["_distances"], q, seed_val)
        fit_results[(row["treebank_id"], q)] = res

logger.info(f"Fitted EVT models for {len(fit_results)} (treebank, threshold) cells")

## Phase 5: Results Assembly and Summary Statistics

In [ ]:
# Assemble per-treebank table
treebank_table = []
for _, row in tb_df.iterrows():
    tb_id = row["treebank_id"]
    entry = {
        "treebank_id": tb_id, "language": row["language"], "family": row["family"],
        "register": row["register"], "n_sentences": int(row["n_sentences"]),
        "n_arcs": int(row["n_arcs"]),
        "head_finality_ratio": row["head_finality_ratio"],
    }
    for q in THRESHOLDS:
        r = fit_results.get((tb_id, q), {})
        suffix = str(int(q * 100))
        entry[f"threshold_{suffix}"] = r.get("threshold")
        entry[f"n_exceedances_{suffix}"] = r.get("n_exceedances")
        entry[f"xi_{suffix}"] = r.get("xi_point")
        entry[f"alpha_{suffix}"] = r.get("alpha_point")
        if q == PRIMARY_THRESHOLD:
            entry["xi_ci_lower"] = r.get("xi_ci_lower")
            entry["xi_ci_upper"] = r.get("xi_ci_upper")
            entry["alpha_ci_lower"] = r.get("alpha_ci_lower")
            entry["alpha_ci_upper"] = r.get("alpha_ci_upper")
    treebank_table.append(entry)

treebank_df = pd.DataFrame(treebank_table)
logger.info(f"Treebank table assembled: {len(treebank_df)} rows")
print(treebank_df[["treebank_id", "language", "register", "xi_75", "alpha_75"]])

## Phase 6: Cross-Treebank Correlation (xi vs alpha)

In [ ]:
def spearman_ci(x, y):
    """Compute Spearman correlation with 95% CI via Fisher z-transform."""
    if len(x) < 4:
        return {"rho": None, "p": None, "ci_lower": None, "ci_upper": None, "n": len(x)}
    rho, p = stats.spearmanr(x, y)
    n = len(x)
    if not np.isfinite(rho) or abs(rho) >= 1.0:
        return {"rho": float(rho), "p": float(p), "ci_lower": None, "ci_upper": None, "n": n}
    z = np.arctanh(rho)
    se = 1.0 / np.sqrt(n - 3)
    lo, hi = z - 1.96 * se, z + 1.96 * se
    return {"rho": float(rho), "p": float(p), "ci_lower": float(np.tanh(lo)),
            "ci_upper": float(np.tanh(hi)), "n": n}

valid = treebank_df.dropna(subset=["xi_75", "alpha_75"])
xi_alpha_corr = spearman_ci(valid["xi_75"].to_numpy(), valid["alpha_75"].to_numpy())
logger.info(f"xi vs alpha Spearman rho={xi_alpha_corr['rho']:.3f} (p={xi_alpha_corr['p']:.4f}, n={xi_alpha_corr['n']})")
print(f"\nxi-alpha correlation: {xi_alpha_corr}")

## Results Summary and Visualization

In [ ]:
# Display key results
print("="*70)
print("DEMO ANALYSIS RESULTS")
print("="*70)
print(f"\n📊 Dataset Summary:")
print(f"  Treebanks analyzed: {len(tb_df)}")
print(f"  Total arcs: {tb_df['n_arcs'].sum()}")
print(f"  Languages: {', '.join(tb_df['language'].unique())}")
print(f"  Registers: {', '.join(tb_df['register'].unique())}")

print(f"\n📈 EVT Tail Index Estimates (at 75th percentile threshold):")
results_table = treebank_df[["treebank_id", "language", "register", "xi_75", "alpha_75"]].copy()
results_table["xi_75"] = results_table["xi_75"].apply(lambda x: f"{x:.4f}" if pd.notna(x) else "—")
results_table["alpha_75"] = results_table["alpha_75"].apply(lambda x: f"{x:.4f}" if pd.notna(x) else "—")
print(results_table.to_string(index=False))

print(f"\n🔗 Cross-Treebank Relationship:")
print(f"  Spearman rho(xi, alpha) = {xi_alpha_corr['rho']:.4f}")
print(f"  p-value = {xi_alpha_corr['p']:.4f}")
print(f"  95% CI = [{xi_alpha_corr['ci_lower']:.4f}, {xi_alpha_corr['ci_upper']:.4f}]")
if xi_alpha_corr['rho'] is not None and abs(xi_alpha_corr['rho']) > 0.7:
    print(f"  → Strong negative correlation: xi and alpha are partially REDUNDANT")
elif xi_alpha_corr['rho'] is not None and abs(xi_alpha_corr['rho']) < 0.5:
    print(f"  → Weak correlation: xi and alpha are RELATIVELY INDEPENDENT")
else:
    print(f"  → Moderate correlation")

print(f"\n⚙️ Configuration Used:")
print(f"  Bootstrap resamples: {B_BOOTSTRAP}")
print(f"  Thresholds analyzed: {THRESHOLDS}")
print(f"  Min arcs (primary): {MIN_ARCS_PRIMARY}")
print(f"\n✅ Demo complete! Scale up parameters for production analysis.")
print("="*70)

## Plot: Tail Shape Comparison

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Plot 1: xi vs alpha scatter
plot_data = treebank_df.dropna(subset=["xi_75", "alpha_75"])
axes[0].scatter(plot_data["xi_75"], plot_data["alpha_75"], s=100, alpha=0.6, color="steelblue")
for idx, row in plot_data.iterrows():
    axes[0].annotate(row["treebank_id"], (row["xi_75"], row["alpha_75"]), 
                     fontsize=8, ha="right", alpha=0.7)
axes[0].set_xlabel("GPD shape (xi)")
axes[0].set_ylabel("Power-law exponent (alpha)")
axes[0].set_title(f"Tail Index Relationship (rho={xi_alpha_corr['rho']:.3f})")
axes[0].grid(True, alpha=0.3)

# Plot 2: Register comparison
reg_data = treebank_df.dropna(subset=["xi_75", "register"])
reg_data["register_num"] = reg_data["register"].map({"written": 0, "spoken": 1})
colors = {"written": "coral", "spoken": "skyblue"}
for reg in ["written", "spoken"]:
    subset = reg_data[reg_data["register"] == reg]
    axes[1].scatter(subset["register_num"], subset["xi_75"], s=100, alpha=0.6, 
                   color=colors[reg], label=reg)
axes[1].set_xticks([0, 1])
axes[1].set_xticklabels(["Written", "Spoken"])
axes[1].set_ylabel("GPD shape (xi)")
axes[1].set_title("Tail Shape by Register")
axes[1].legend()
axes[1].grid(True, alpha=0.3, axis="y")

plt.tight_layout()
plt.show()
logger.info("Plots generated")